# OcuScan AI — Evaluation Notebook
**Phase 3 | Week 3 | Day 21**

Complete evaluation report:
- Full 6-class metrics (accuracy, macro F1, per-class P/R/F1)
- Confusion matrix with OCP ↔ OCP Chronic highlighting
- AUC-ROC curves (per-class + macro)
- Calibration reliability diagram + ECE
- Temperature scaling (if needed)
- 5-fold cross-validation results
- SVM baseline comparison
- Grad-CAM samples (OCP ×2, OCP Chronic ×2, SJS ×1, Symblepharon ×1)


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from evaluate import (
    compute_all_metrics,
    plot_confusion_matrix,
    plot_roc_curves,
    plot_calibration_curve,
    print_metrics_table,
    run_cross_validation,
    run_svm_baseline,
    TemperatureScaler,
    CLASS_NAMES,
    DISPLAY_NAMES,
    RESULTS_DIR,
)
from gradcam import GradCAM, generate_phase3_required_heatmaps, GRADCAM_DIR
from utils import format_metrics_summary

# --- Configuration ---
LABELS_CSV     = '../dataset/labels.csv'
CHECKPOINT     = '../models/phase2_best.pt'
DATASET_DIR    = '../dataset'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE     = 8

print(f'Device: {DEVICE}')
print(f'Results dir: {RESULTS_DIR.resolve()}')

---
## 1. Load Model & Run Test Set Inference

In [ ]:
from model import OcuScanModel
from dataset import EyeDiseaseDataset
from augmentation import val_transform

model = OcuScanModel(num_classes=len(CLASS_NAMES))
model.load_checkpoint(CHECKPOINT)
model.to(DEVICE)
model.eval()
print(f'Loaded checkpoint: {CHECKPOINT}')

df = pd.read_csv(LABELS_CSV)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
train_df = df[df['split'] == 'train'].reset_index(drop=True)

print(f'Test samples  : {len(test_df)}')
print(f'Val samples   : {len(val_df)}')
print(f'Train samples : {len(train_df)}')
print()
print('Test class distribution:')
print(test_df['class_key'].value_counts().to_string())

In [ ]:
def run_inference_on_split(subset_df, device=DEVICE, batch_size=BATCH_SIZE):
    dataset = EyeDiseaseDataset(
        filepaths=subset_df['filepath'].tolist(),
        labels=subset_df['class_idx'].tolist(),
        transform=val_transform,
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_logits, all_true = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            logits = model(imgs.to(device))
            all_logits.append(logits.cpu().numpy())
            all_true.extend(lbls.numpy())
    return np.vstack(all_logits), np.array(all_true)

print('Running test set inference...')
test_logits, y_true = run_inference_on_split(test_df)
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
y_pred = test_probs.argmax(axis=1)
print(f'Done. Shape: logits={test_logits.shape}, labels={y_true.shape}')

---
## 2. Core Metrics

In [ ]:
metrics = compute_all_metrics(y_true, y_pred, test_probs, CLASS_NAMES)
print_metrics_table(metrics)

# Save metrics.json
import json
with open(RESULTS_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('\nmetrics.json saved.')

In [ ]:
# Pretty per-class table
rows = []
for key, v in metrics['per_class'].items():
    rows.append({
        'Class': v['display'],
        'Precision': v['precision'],
        'Recall': v['recall'],
        'F1': v['f1'],
        'Support': v['support'],
        'AUC-ROC': metrics['auc_roc_per_class'].get(key, 'N/A'),
    })
per_class_df = pd.DataFrame(rows)
display(per_class_df.style.background_gradient(subset=['F1', 'AUC-ROC'], cmap='Blues'))

---
## 3. Confusion Matrix (with OCP ↔ OCP Chronic Highlighting)

In [ ]:
plot_confusion_matrix(y_true, y_pred)
img = mpimg.imread(RESULTS_DIR / 'confusion_matrix.png')
fig, ax = plt.subplots(figsize=(11, 9))
ax.imshow(img)
ax.axis('off')
plt.title('Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

ocp = metrics['ocp_vs_ocp_chronic']
print(f"OCP confusion rate          : {ocp['ocp_confusion_rate']:.4f}")
print(f"OCP predicted as OCP Chronic: {ocp['ocp_confused_as_chronic']}")
print(f"OCP Chronic predicted as OCP: {ocp['ocp_chronic_confused_as_ocp']}")

---
## 4. AUC-ROC Curves

In [ ]:
plot_roc_curves(y_true, test_probs)
img = mpimg.imread(RESULTS_DIR / 'roc_curves.png')
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Macro AUC-ROC:', metrics['macro_auc_roc'])
print('Per-class AUC:')
for k, v in metrics['auc_roc_per_class'].items():
    print(f'  {DISPLAY_NAMES[k]:<30}: {v}')

---
## 5. Calibration Curves

In [ ]:
ece_scores = plot_calibration_curve(y_true, test_probs)
img = mpimg.imread(RESULTS_DIR / 'calibration_curve.png')
fig, ax = plt.subplots(figsize=(14, 9))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Expected Calibration Error (ECE) per class:')
for k, v in ece_scores.items():
    print(f'  {DISPLAY_NAMES[k]:<30}: {v}')

In [ ]:
# ─── Temperature Scaling ────────────────────────────────────────────────────
# Only apply if model is noticeably overconfident (ECE > 0.1 on average)

mean_ece = np.mean([v for v in ece_scores.values() if v is not None])
print(f'Mean ECE: {mean_ece:.4f}')

if mean_ece > 0.1:
    print('ECE > 0.1: fitting temperature scaling on val set...')
    val_logits, val_true = run_inference_on_split(val_df)
    scaler = TemperatureScaler()
    T = scaler.fit(val_logits, val_true)
    print(f'Fitted temperature: {T:.4f}')

    cal_probs = scaler.transform(test_logits)
    cal_pred  = cal_probs.argmax(axis=1)
    ece_after = plot_calibration_curve(
        y_true, cal_probs,
        save_path=RESULTS_DIR / 'calibration_curve_after_scaling.png'
    )
    mean_ece_after = np.mean([v for v in ece_after.values() if v is not None])
    print(f'Mean ECE after scaling: {mean_ece_after:.4f}')

    # Save scaler
    import json
    with open(RESULTS_DIR / 'temperature_scaler.json', 'w') as f:
        json.dump({'temperature': T}, f)
    print('Temperature scaler saved.')
else:
    print('ECE acceptable — temperature scaling not required.')

---
## 6. 5-Fold Cross-Validation

In [ ]:
cv_results = run_cross_validation(
    labels_csv=LABELS_CSV,
    checkpoint_path=CHECKPOINT,
    device=DEVICE,
    n_splits=5,
)

print(f"\n5-Fold CV Summary")
print(f"Macro F1: {cv_results['macro_f1_mean']:.4f} ± {cv_results['macro_f1_std']:.4f}")
print(f"Per-fold: {cv_results['macro_f1_folds']}")
print()
print('Per-class F1 (mean ± std):')
for k, v in cv_results['per_class_f1'].items():
    print(f"  {DISPLAY_NAMES[k]:<30}: {v['mean']:.4f} ± {v['std']:.4f}")

In [ ]:
# CV results bar chart
labels_disp = [DISPLAY_NAMES[k] for k in CLASS_NAMES]
means  = [cv_results['per_class_f1'][k]['mean'] for k in CLASS_NAMES]
stds   = [cv_results['per_class_f1'][k]['std']  for k in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels_disp, means, yerr=stds, capsize=5, color='#3b82f6', alpha=0.85, edgecolor='white')
ax.axhline(cv_results['macro_f1_mean'], color='#ef4444', linewidth=2, linestyle='--',
           label=f"Macro F1 = {cv_results['macro_f1_mean']:.4f} ± {cv_results['macro_f1_std']:.4f}")
ax.set_ylim([0, 1.05])
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('5-Fold CV — Per-Class F1 (mean ± std)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cv_f1_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. SVM Baseline Comparison

In [ ]:
svm_results = run_svm_baseline(
    labels_csv=LABELS_CSV,
    checkpoint_path=CHECKPOINT,
    device=DEVICE,
)

print('\nSVM Baseline vs Fine-Tuned EfficientNetB0 Comparison')
print('='*60)
comparison = []
for key in CLASS_NAMES:
    finetune_f1 = metrics['per_class'][key]['f1']
    svm_f1      = svm_results['per_class'][key]['f1']
    delta       = finetune_f1 - svm_f1
    comparison.append({
        'Class'          : DISPLAY_NAMES[key],
        'Fine-tuned F1'  : finetune_f1,
        'SVM F1'         : svm_f1,
        'Δ F1'           : round(delta, 4),
    })

cmp_df = pd.DataFrame(comparison)
print(cmp_df.to_string(index=False))
print()
print(f"Macro F1 — Fine-tuned: {metrics['macro_f1']}  |  SVM: {svm_results['macro_f1']}  |  Δ={round(metrics['macro_f1']-svm_results['macro_f1'],4)}")

In [ ]:
# Comparison bar chart
x = np.arange(len(CLASS_NAMES))
w = 0.35
finetune_f1s = [metrics['per_class'][k]['f1'] for k in CLASS_NAMES]
svm_f1s      = [svm_results['per_class'][k]['f1'] for k in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, finetune_f1s, w, label='Fine-tuned EfficientNetB0', color='#2563eb', alpha=0.85)
ax.bar(x + w/2, svm_f1s,      w, label='SVM on frozen features',    color='#f59e0b', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAMES[k] for k in CLASS_NAMES], rotation=20, ha='right', fontsize=9)
ax.set_ylim([0, 1.05])
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('Fine-tuned EfficientNetB0 vs SVM Baseline — Per-Class F1', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'svm_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Grad-CAM Heatmaps
Required: 2× OCP, 2× OCP Chronic, 1× SJS, 1× Symblepharon

Visual validation: confirm Symblepharon adhesion band is highlighted.

In [ ]:
saved_paths = generate_phase3_required_heatmaps(
    model=model,
    dataset_dir=DATASET_DIR,
    save_dir=GRADCAM_DIR,
)
print(f'\n{len(saved_paths)} Grad-CAM images generated:')
for p in saved_paths:
    print(f'  {p}')

In [ ]:
# Display all generated heatmaps inline
n = len(saved_paths)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, path in zip(axes, saved_paths):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(Path(path).stem, fontsize=7)
    plt.suptitle('Grad-CAM Overlays — Phase 3 Required Samples', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No Grad-CAM images generated — check dataset folder structure.')

In [ ]:
# ─── Visual Validation Notes ────────────────────────────────────────────────
validation_notes = {
    'ocp'          : 'VERIFY: subconjunctival fibrosis / forniceal region highlighted',
    'ocp_chronic'  : 'VERIFY: dense fibrosis / advanced forniceal loss region highlighted',
    'sjs'          : 'VERIFY: pseudomembrane or keratinisation region highlighted',
    'symblepharon' : 'VERIFY: fibrous adhesion band between palpebral and bulbar conjunctiva highlighted',
}
print('Manual validation checklist:')
for k, note in validation_notes.items():
    print(f'  [{DISPLAY_NAMES[k]:<28}] {note}')

---
## 9. Full Results Summary

In [ ]:
print(format_metrics_summary(RESULTS_DIR / 'metrics.json'))

print('\n--- Results folder ---')
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        size_kb = f.stat().st_size // 1024
        print(f'  {str(f.relative_to(RESULTS_DIR)):<45} {size_kb:>5} KB')

In [ ]:
# ─── Phase 3 Completion Checklist ───────────────────────────────────────────
results_dir = RESULTS_DIR
checklist = {
    'Day 15 — evaluate.py runs on test set'   : True,  # evaluated above
    'Day 15 — metrics.json committed'         : (results_dir / 'metrics.json').exists(),
    'Day 16 — confusion_matrix.png saved'     : (results_dir / 'confusion_matrix.png').exists(),
    'Day 16 — ocp_confusion_rate logged'      : 'ocp_vs_ocp_chronic' in metrics,
    'Day 17 — roc_curves.png saved'           : (results_dir / 'roc_curves.png').exists(),
    'Day 17 — calibration_curve.png saved'    : (results_dir / 'calibration_curve.png').exists(),
    'Day 18 — cv_results.json committed'      : (results_dir / 'cv_results.json').exists(),
    'Day 19 — svm_baseline.json committed'    : (results_dir / 'svm_baseline.json').exists(),
    'Day 20 — 6 Grad-CAM samples saved'       : len(saved_paths) >= 6,
    'Day 21 — Notebook fully executed'        : True,
}

all_pass = all(checklist.values())
print('Phase 3 Completion Checklist')
print('=' * 50)
for item, status in checklist.items():
    icon = '✅' if status else '❌'
    print(f'  {icon}  {item}')
print()
print('Phase 3 STATUS:', '✅ COMPLETE' if all_pass else '⚠ INCOMPLETE — review ❌ items above')